# 1105 45일차

## 1. OpenAI 호환 API

NPU에 올린 모델에 OpenAI 라이브러리로 붙어서 추론을 요청하는 방법

`furiosa-llm serve`로 띄운 서버가 OpenAI API와 같은 경로·요청 형식을 받아서, 주소만 바꾸면 OpenAI 모델을 쓰던 코드를 그대로 씀

```
furiosa-llm serve furiosa-ai/Qwen3-4B-FP8 \
    --devices npu:6 \
    --host 0.0.0.0 \
    --port 8000
```

1. `--devices npu:6` : 모델을 올릴 장치 번호
2. `--host 0.0.0.0` : 요청을 받을 주소
3. `--port 8000` : 요청을 받을 포트

```python
client = OpenAI(base_url="http://127.0.0.1:8000/v1", api_key="EMPTY")
model_id = client.models.list().data[0].id
```

- `npu:6`의 번호는 내가 고른 장치. `furiosa-smi status`로 비어 있는 장치를 보고 정함
- `--host 0.0.0.0`은 어느 네트워크 쪽에서 와도 받고, `127.0.0.1`은 같은 기계에서 온 요청만 받음
- `base_url`은 `/v1`까지 적음. OpenAI가 쓰는 경로 규격을 서버가 그대로 받기 때문
- `api_key`는 로컬 서버가 보지 않지만 라이브러리가 빈 값을 거부해서 `EMPTY`처럼 아무 문자열을 넣음
- `client.models`는 요청을 보내는 리소스 객체라 그 자체에는 `data`가 없음. `.list()`를 불러야 서버에 요청이 가고 그 응답에 `data` 배열이 들어 있음
- `data`는 서버에 올라간 모델 목록. 이름을 받아서 쓰면 모델을 바꿀 때 코드를 고치지 않음

## 2. ChatOpenAI

LangChain에서 같은 서버에 붙는 방법

체인이나 에이전트는 LangChain 쪽 객체를 받으므로, NPU 서버를 그 구성에 그대로 끼워 쓰려고 씀

1. `pip install langchain-openai` : OpenAI 호환 서버용 LangChain 패키지
2. `ChatOpenAI(base_url=…, api_key=…)` : 주소를 로컬 서버로 돌림
3. `llm.invoke("…")` : 질문 문자열을 넘겨 호출
4. `response.content` : 답변 본문

```python
llm = ChatOpenAI(base_url="http://127.0.0.1:8000/v1", api_key="EMPTY")
response = llm.invoke("NPU와 GPU의 차이점을 설명해줘")
```

- OpenAI 라이브러리는 `messages` 목록에 `role`과 `content`를 직접 만들어 넘기고, `ChatOpenAI`는 문자열 하나를 넘기면 사용자 메시지로 감싸서 보냄
- 답변을 꺼내는 자리도 다름. OpenAI 라이브러리는 `choices[0].message.content`, LangChain은 `content`
- `max_tokens`·`temperature`·`top_p` 같은 생성 옵션은 `ChatOpenAI`를 만들 때 넣어 두면 호출마다 다시 적지 않음

## 3. 처리량 측정

서버가 1초에 토큰을 몇 개 만드는지 재는 것

응답 시간만 보면 답변이 길어서 오래 걸린 것인지 서버가 느려서 오래 걸린 것인지 구분되지 않음. 만든 토큰 수로 나눠야 모델과 장치를 같은 기준으로 비교함

1. `max_tokens=1000` : 생성할 토큰 수를 고정
2. `time.perf_counter()` : 호출 전후 시각의 차이로 걸린 시간
3. `max_tokens / elapsed` : 초당 토큰 수(tokens/s)

```python
start = time.perf_counter()
response = llm.invoke("NPU와 GPU의 차이점을 설명해줘")
elapsed = round(time.perf_counter() - start, 2)
```

- `perf_counter`는 날짜·시각이 아니라 흐른 시간만 재는 함수. 시스템 시각이 바뀌어도 측정값이 흔들리지 않음
- 요청마다 답변 길이가 달라지면 수치를 견줄 수 없음. **`max_tokens`를 고정한 상태에서만 처리량을 비교함**
- 모델이 `max_tokens`보다 먼저 답을 끝냈는데 `max_tokens`로 나누면 처리량이 실제보다 높게 나옴
- `temperature`는 다음 토큰을 고를 때 주는 무작위성. 0이면 늘 같은 답, 높이면 답이 매번 달라짐
- `top_p`는 확률 높은 순으로 누적해서 그 비율 안에 드는 토큰만 후보로 둠
- 처리량은 모델 크기·자료형·장치 수에 따라 달라지므로, 견줄 때는 하나만 바꾸고 나머지는 그대로 둠

## 4. 스트리밍

답변이 다 만들어질 때까지 기다리지 않고 토큰이 나오는 대로 받아 출력하는 것

긴 답변은 완성까지 몇 초가 걸려서 그동안 화면에 아무것도 없음. 조각으로 받으면 첫 글자가 나오는 시점이 앞당겨짐

1. `stream=True` : 응답을 한 번에 받지 않고 조각으로 받음
2. `chunk.choices[0].delta.content` : 그 조각에 새로 담긴 글자
3. `print(content, end="", flush=True)` : 줄바꿈 없이 바로 화면에 씀

```python
for chunk in streaming:
    if chunk.choices:
        content = chunk.choices[0].delta.content
        if content:
            print(content, end="", flush=True)
```

- 스트리밍이 아니면 `message.content`에 답변 전체가 한 번에 들어오고, 스트리밍은 `delta`에 직전 조각 다음에 붙는 부분만 들어옴
- 반환값은 완성된 응답이 아니라 반복할 수 있는 대상이라 `for`로 돌림
- `if chunk.choices:`는 본문 없이 메타데이터만 담긴 조각을 건너뛰려고 둠
- `content`가 `None`인 조각도 있어서 한 번 더 걸러야 `print`에 `None`이 찍히지 않음
- `end=""`는 `print`가 붙이는 줄바꿈을 없애서 조각을 이어 붙임
- `flush=True`는 버퍼에 모으지 않고 바로 내보냄. 없으면 조각이 쌓였다가 한꺼번에 나옴
- 전체 걸리는 시간은 같고 첫 글자가 나오는 시점만 빨라짐

#### cf) LangChain으로 스트리밍

```python
for chunk in llm.stream("NPU랑 GPU의 차이점을 설명해줘"):
    print(chunk.content, end="", flush=True)
```

- `llm.stream`도 조각을 하나씩 돌려줌. 꺼내는 자리가 `content`라 `choices`·`delta`를 거치지 않음
- 본문 없는 조각을 걸러 주므로 `if`도 필요 없음
- `end=""`·`flush=True`는 그대로 필요함. 화면에 쓰는 쪽 사정이라 어느 라이브러리를 쓰든 같음

## 5. 공통 설정 모듈

같은 설정을 쓰는 스크립트가 여러 개일 때, 설정을 한 파일에 모아 두고 불러 쓰는 것

스크립트마다 주소와 생성 옵션을 적으면 서버가 바뀔 때 전부 고쳐야 함. 객체를 만드는 코드를 함수 하나로 빼면 한 군데만 고침

1. `common.py` : `ChatOpenAI`를 만들어 돌려주는 `get_llm` 함수를 둠
2. `def get_llm(max_tokens=1000, temperature=0, top_p=0.9):` : 자주 쓰는 값을 기본값으로
3. `from common import get_llm` : 쓰는 쪽에서 불러옴
4. `get_llm(temperature=0.9, top_p=1.0)` : 바꿀 값만 적어 호출

- 기본값이 있으면 `get_llm()`처럼 인자를 안 넘겨도 되고, 넘긴 것만 덮어씀
- `from common import get_llm`의 `common`은 파일 이름에서 `.py`를 뗀 모듈 이름
- 같은 폴더에 있으면 경로를 적지 않아도 찾음. 파이썬이 실행하는 스크립트가 있는 폴더를 먼저 뒤지기 때문
- 함수가 객체를 만들어 돌려주므로 부를 때마다 새 `llm`이 생김
- **서버 주소나 모델을 바꿀 일이 생기면 `common.py` 한 군데만 고침**